# Decoder training

Trains a decoder on one saved calibration block, estimates its accuracy with cross-validation and saves it for the online decoder.
The decoder parameters are in `bci/config.py`. Autoreload is on: after editing a file in `bci/`, re-running a cell uses the new version without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from bci import config, plot
from bci.decoder import train_decoder
from bci.preprocessing import bandpass, load_block, sliding_windows, task_epochs

## Block and model files

- `BLOCK_PATH`: saved block used for training.
- `MODEL_PATH`: file of the saved model, named after the block and the decoder.

Decoder parameters in `bci/config.py`:
- `DECODER_NAME`: key of `DECODERS` in `bci/decoder.py`.
- `DECODER_CLASSES`: classes the model learns; trials of other classes are left out (e.g. `["right", "rest"]` for one leg against rest).
- `DECODER_BAND_HZ`, `DECODER_FILTER_ORDER`, `WINDOW_S`: saved with the model and reused by the online decoder.
- `WINDOW_STEP_S`, `TRAINING_START_S`, `TRAINING_END_S`: which windows of each imagery period are used for training.
- `FOLD_COUNT`: number of cross-validation folds; each class needs at least this many trials.

In [ ]:
BLOCK_PATH = config.SOURCEDATA_DIR / "sub-p001/ses-2/eeg/sub-p001_ses-2_task-calibration_run-001_eeg.xdf"
MODEL_PATH = config.MODELS_DIR / f"{BLOCK_PATH.stem}_{config.DECODER_NAME}.pkl"

## Training windows

The continuous EEG is filtered with the same causal band-pass used online, cut into one epoch per imagery period (labelled with the preceding cue) and then into overlapping windows of `WINDOW_S` seconds.
Windows have shape windows x channels x samples, the layout expected by pyriemann.

In [ ]:
eeg, eeg_times, marker_codes, marker_times = load_block(BLOCK_PATH)
filtered_eeg, _ = bandpass(eeg, config.EEG_SAMPLING_RATE_HZ, config.DECODER_BAND_HZ[0], config.DECODER_BAND_HZ[1],
                        config.DECODER_FILTER_ORDER)
epochs, labels, _ = task_epochs(filtered_eeg, eeg_times, marker_codes, marker_times,
                                config.TRAINING_START_S, config.TRAINING_END_S)
kept = np.isin(labels, config.DECODER_CLASSES)
window_samples = int(round(config.WINDOW_S * config.EEG_SAMPLING_RATE_HZ))
step_samples = int(round(config.WINDOW_STEP_S * config.EEG_SAMPLING_RATE_HZ))
windows, window_labels, window_trials = sliding_windows(epochs[kept], labels[kept], window_samples, step_samples)
print(f"{len(windows)} windows from {kept.sum()} trials, shape {windows.shape}")

## Cross-validation, training and saving

Windows of the same trial overlap, so they always stay in the same fold; otherwise the accuracy would be inflated.
Each window is predicted by a decoder trained on the other folds; then the decoder is fitted on all windows and saved to `MODEL_PATH` together with `settings`.
Chance level is 1 / number of classes. The confusion matrix shows, for each true class, the fraction of windows assigned to each predicted class.

In [ ]:
settings = {"band_hz": config.DECODER_BAND_HZ, "filter_order": config.DECODER_FILTER_ORDER,
            "window_s": config.WINDOW_S, "sampling_rate_hz": config.EEG_SAMPLING_RATE_HZ}
predictions = train_decoder(config.DECODER_NAME, windows, window_labels, window_trials, config.FOLD_COUNT,
                            MODEL_PATH, settings)
print(f"cross-validated window accuracy: {np.mean(predictions == window_labels):.2f}")
print(f"model saved to {MODEL_PATH}")
plot.plot_confusion_matrix(window_labels, predictions)